# Homework 2

This notebook contains the code and answer(s) for Problem 2 of Homework 2.

---

Problem 2.) In class we talked about the 3rd and X model.  Using data from the most recent 5 NFL seasons, fit the model described in class and in a well written paragraph evaluate the claim that ``all teams in the NFL are equally good at making 3rd and X.''  Explain any choices that you made regarding the data and the models.

In [84]:
# Import the neccesary Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import nfl_data_py as nfl
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
from sklearn.model_selection import cross_val_score, StratifiedKFold

In [85]:
# Import the data
nfl_data = nfl.import_pbp_data(years=[2021, 2022, 2023, 2024, 2025])

2021 done.
2022 done.
2023 done.
2024 done.
2025 done.
Downcasting floats.


---

## Notes from class:

**Data:** pbp data, 3rd down data, filter out garbage time, filter out 3rd downs with penalties that move the offense back, filter out end of half data, filter out long 3rd down data (x >= sum #), filter out time remaining in half < 30 secs, filter out own rz, filter out close games (fg takes lead on ‘change in wp’), filter out playoff games, (maybe filter out week 18 - good idea but maybe leave out. This is a ‘choices’ thing).

**(The above are all ideas for filter)**

**Target:** Completed or not

From ‘completed’, it could mean 1st score, or a score, penalties by the defense, 4th down conversion

**predictors/ features:** x = yards to go, offensive/pos team

(above are the HAVE TO HAVE NO MATTER WHAT predictors)

**Form:** linear offense + s(x) + …

s(x) = smooth function

**Modeling choice:** logistic regression (other classification models possible - maybe classification trees but probably not SVM’s -)

**Other predictors:** Defense, home field or not

Likely only do this on data from one year only due to player and coaching changes on certain teams.

Using the notes from above, 2 models will be built. 

- A model with only the 'MUST HAVE' predictors (yards to go (the big X) and offensive pos team)
- And a model including the 'MUST HAVE' predictors as well as both of the other predictors (Defensive pos team and home field advantage for an offense or not).
- If the model that includes both of the other predictors has one variable that is significant and the other is not, then the insignificant predictor will be removed and a 3rd model with the 2 neccesary predictors and the significant optional variable will be tested.

Note: The third bullet point is what occured with posteam_type being significant and the defensive teams being insignifcant.

Additional note about variables:

- posteam = offensive team
- defteam = defensive team
- first_down = result of play is first down or not
- posteam_type = home field advantage or not for the offense (operation to edit this variable is in the section below)
- ydstogo = yards to go until first down line before the play


---

## Data Filtering

Before modeling the data, I will need to filter the plays to make sure that all of the metrics/ flags we want to use are not null and that they occur on 3rd down.



In [86]:
# Examine the available columns of the play by play data
col_list = nfl_data.columns.tolist()
pd.set_option('display.max_seq_items', None)
print("\n".join(nfl_data.columns))

play_id
game_id
old_game_id_x
home_team
away_team
season_type
week
posteam
posteam_type
defteam
side_of_field
yardline_100
game_date
quarter_seconds_remaining
half_seconds_remaining
game_seconds_remaining
game_half
quarter_end
drive
sp
qtr
down
goal_to_go
time
yrdln
ydstogo
ydsnet
desc
play_type
yards_gained
shotgun
no_huddle
qb_dropback
qb_kneel
qb_spike
qb_scramble
pass_length
pass_location
air_yards
yards_after_catch
run_location
run_gap
field_goal_result
kick_distance
extra_point_result
two_point_conv_result
home_timeouts_remaining
away_timeouts_remaining
timeout
timeout_team
td_team
td_player_name
td_player_id
posteam_timeouts_remaining
defteam_timeouts_remaining
total_home_score
total_away_score
posteam_score
defteam_score
score_differential
posteam_score_post
defteam_score_post
score_differential_post
no_score_prob
opp_fg_prob
opp_safety_prob
opp_td_prob
fg_prob
safety_prob
td_prob
extra_point_prob
two_point_conversion_prob
ep
epa
total_home_epa
total_away_epa
total_home_rush_ep

In [87]:
# Filtering process
nfl_data = nfl_data[
    (nfl_data['down'] == 3) &
    (nfl_data['ydstogo'].notna()) &
    (nfl_data['posteam'].notna()) &
    (nfl_data['defteam'].notna()) &
    (nfl_data['posteam_type'].notna()) &
    (nfl_data['first_down'].notna())
]

In [88]:
# Need to make posteam_type into 1s and 0s.
nfl_data['posteam_type'] = nfl_data['posteam_type'].map({'home': 1, 'away': 0})

We will also need to make dummy for both the posteam (offensive team) and the defteam (defensive team)

In [89]:
# Get dummies for posteam and defteam
nfl_data = pd.get_dummies(data=nfl_data, columns=['posteam', 'defteam'], dtype=int)

In [90]:
# Look at the results
# Examine the available columns of the play by play data
nfl_data.columns[395:]

Index(['posteam_ARI', 'posteam_ATL', 'posteam_BAL', 'posteam_BUF',
       'posteam_CAR', 'posteam_CHI', 'posteam_CIN', 'posteam_CLE',
       'posteam_DAL', 'posteam_DEN', 'posteam_DET', 'posteam_GB',
       'posteam_HOU', 'posteam_IND', 'posteam_JAX', 'posteam_KC', 'posteam_LA',
       'posteam_LAC', 'posteam_LV', 'posteam_MIA', 'posteam_MIN', 'posteam_NE',
       'posteam_NO', 'posteam_NYG', 'posteam_NYJ', 'posteam_PHI',
       'posteam_PIT', 'posteam_SEA', 'posteam_SF', 'posteam_TB', 'posteam_TEN',
       'posteam_WAS', 'defteam_ARI', 'defteam_ATL', 'defteam_BAL',
       'defteam_BUF', 'defteam_CAR', 'defteam_CHI', 'defteam_CIN',
       'defteam_CLE', 'defteam_DAL', 'defteam_DEN', 'defteam_DET',
       'defteam_GB', 'defteam_HOU', 'defteam_IND', 'defteam_JAX', 'defteam_KC',
       'defteam_LA', 'defteam_LAC', 'defteam_LV', 'defteam_MIA', 'defteam_MIN',
       'defteam_NE', 'defteam_NO', 'defteam_NYG', 'defteam_NYJ', 'defteam_PHI',
       'defteam_PIT', 'defteam_SEA', 'defteam_SF', 'd

---

### Model with ONLY 'yards to go' and 'offensive pos team':

In [91]:
# Get the predictors
X = nfl_data[['ydstogo', 'posteam_ARI', 'posteam_ATL', 'posteam_BAL', 'posteam_BUF',
       'posteam_CAR', 'posteam_CHI', 'posteam_CIN', 'posteam_CLE',
       'posteam_DAL', 'posteam_DEN', 'posteam_DET', 'posteam_GB',
       'posteam_HOU', 'posteam_IND', 'posteam_JAX', 'posteam_KC', 'posteam_LA',
       'posteam_LAC', 'posteam_LV', 'posteam_MIA', 'posteam_MIN', 'posteam_NE',
       'posteam_NO', 'posteam_NYG', 'posteam_NYJ', 'posteam_PHI',
       'posteam_PIT', 'posteam_SEA', 'posteam_SF', 'posteam_TB', 'posteam_TEN',
       'posteam_WAS']]

# Get the target predictor
y = nfl_data[['first_down']]

# test train split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Make the model object and train it
model_1 = sm.Logit(y_train, X_train).fit()

# Test the model
pred_prob = model_1.predict(X_test)
pred_class = (pred_prob >= 0.5).astype(int)

# Get the summary
model_1.summary()

Optimization terminated successfully.
         Current function value: 0.615857
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:             first_down   No. Observations:                32040
Model:                          Logit   Df Residuals:                    32007
Method:                           MLE   Df Model:                           32
Date:                Sat, 29 Aug 2026   Pseudo R-squ.:                 0.08332
Time:                        17:13:23   Log-Likelihood:                -19732.
converged:                       True   LL-Null:                       -21525.
Covariance Type:            nonrobust   LLR p-value:                     0.000
===============================================================================
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
ydstogo        -0.1596      0.003    -52.366      0.000      -0.166      -0.154
posteam_ARI     0.6930      0.071      9.708      0.000       0.553       0.833
posteam_ATL     0.5354      0.073      7.369      0.000       0.393       0.678
posteam_BAL     0.6441      0.071      9.113      0.000       0.506       0.783
posteam_BUF     0.8615      0.069     12.523      0.000       0.727       0.996
posteam_CAR     0.3472      0.073      4.746      0.000       0.204       0.491
posteam_CHI     0.6088      0.069      8.842      0.000       0.474       0.744
posteam_CIN     0.7573      0.069     10.959      0.000       0.622       0.893
posteam_CLE     0.4955      0.070      7.072      0.000       0.358       0.633
posteam_DAL     0.7799      0.068     11.540      0.000       0.647       0.912
posteam_DEN     0.5507      0.070      7.881      0.000       0.414       0.688
posteam_DET     0.6324      0.072      8.791      0.000       0.491       0.773
posteam_GB      0.7041      0.071      9.912      0.000       0.565       0.843
posteam_HOU     0.5497      0.069      7.932      0.000       0.414       0.686
posteam_IND     0.5081      0.072      7.047      0.000       0.367       0.649
posteam_JAX     0.5519      0.072      7.715      0.000       0.412       0.692
posteam_KC      0.8358      0.068     12.323      0.000       0.703       0.969
posteam_LA      0.5621      0.070      8.057      0.000       0.425       0.699
posteam_LAC     0.6868      0.068     10.043      0.000       0.553       0.821
posteam_LV      0.5029      0.073      6.864      0.000       0.359       0.646
posteam_MIA     0.5747      0.074      7.803      0.000       0.430       0.719
posteam_MIN     0.5808      0.073      7.995      0.000       0.438       0.723
posteam_NE      0.4403      0.072      6.098      0.000       0.299       0.582
posteam_NO      0.5751      0.072      7.995      0.000       0.434       0.716
posteam_NYG     0.5301      0.070      7.554      0.000       0.393       0.668
posteam_NYJ     0.4498      0.074      6.106      0.000       0.305       0.594
posteam_PHI     0.7036      0.067     10.488      0.000       0.572       0.835
posteam_PIT     0.6504      0.068      9.549      0.000       0.517       0.784
posteam_SEA     0.5089      0.074      6.864      0.000       0.364       0.654
posteam_SF      0.8433      0.070     11.987      0.000       0.705       0.981
posteam_TB      0.6913      0.068     10.224      0.000       0.559       0.824
posteam_TEN     0.5067      0.072      7.037      0.000       0.366       0.648
posteam_WAS     0.5935      0.071      8.405      0.000       0.455       0.732
===============================================================================
"""

In [92]:
# print out other classification accuracy metrics
class_report = classification_report(y_test, pred_class)
conf_matrix = confusion_matrix(y_test, pred_class)
roc_auc = roc_auc_score(y_test, pred_prob)
accuracy = accuracy_score(y_test, pred_class)

print(f'Classification Report: \n {class_report}')
print(f'Confusion Matrix: \n {conf_matrix}')
print(f'roc_auc_score: {roc_auc}')
print(f'Accuracy: {accuracy}')

Classification Report: 
               precision    recall  f1-score   support

         0.0       0.69      0.77      0.73      4841
         1.0       0.57      0.46      0.51      3170

    accuracy                           0.65      8011
   macro avg       0.63      0.62      0.62      8011
weighted avg       0.64      0.65      0.64      8011

Confusion Matrix: 
 [[3749 1092]
 [1709 1461]]
roc_auc_score: 0.6847868854168229
Accuracy: 0.6503557608288603


---

## Model with BOTH extra predictors 

In [93]:
# Get the predictors
X = nfl_data[['ydstogo', 'posteam_ARI', 'posteam_ATL', 'posteam_BAL', 'posteam_BUF',
       'posteam_CAR', 'posteam_CHI', 'posteam_CIN', 'posteam_CLE',
       'posteam_DAL', 'posteam_DEN', 'posteam_DET', 'posteam_GB',
       'posteam_HOU', 'posteam_IND', 'posteam_JAX', 'posteam_KC', 'posteam_LA',
       'posteam_LAC', 'posteam_LV', 'posteam_MIA', 'posteam_MIN', 'posteam_NE',
       'posteam_NO', 'posteam_NYG', 'posteam_NYJ', 'posteam_PHI',
       'posteam_PIT', 'posteam_SEA', 'posteam_SF', 'posteam_TB', 'posteam_TEN',
       'posteam_WAS',
       'posteam_type',
       'defteam_ARI', 'defteam_ATL', 'defteam_BAL',
       'defteam_BUF', 'defteam_CAR', 'defteam_CHI', 'defteam_CIN',
       'defteam_CLE', 'defteam_DAL', 'defteam_DEN', 'defteam_DET',
       'defteam_GB', 'defteam_HOU', 'defteam_IND', 'defteam_JAX', 'defteam_KC',
       'defteam_LA', 'defteam_LAC', 'defteam_LV', 'defteam_MIA', 'defteam_MIN',
       'defteam_NE', 'defteam_NO', 'defteam_NYG', 'defteam_NYJ', 'defteam_PHI',
       'defteam_PIT', 'defteam_SEA', 'defteam_SF', 'defteam_TB', 'defteam_TEN',
       'defteam_WAS']]

# Get the target predictor
y = nfl_data[['first_down']]

# test train split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Make the model object and train it
model_2 = sm.Logit(y_train, X_train).fit()

# Test the model
pred_prob = model_2.predict(X_test)
pred_class = (pred_prob >= 0.5).astype(int)

# Get the summary
model_2.summary()

Optimization terminated successfully.
         Current function value: 0.615366
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:             first_down   No. Observations:                32040
Model:                          Logit   Df Residuals:                    31975
Method:                           MLE   Df Model:                           64
Date:                Sat, 29 Aug 2026   Pseudo R-squ.:                 0.08405
Time:                        17:13:24   Log-Likelihood:                -19716.
converged:                       True   LL-Null:                       -21525.
Covariance Type:            nonrobust   LLR p-value:                     0.000
================================================================================
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
ydstogo         -0.1597      0.003    -52.302      0.000      -0.166      -0.154
posteam_ARI      0.3845   9.04e+05   4.25e-07      1.000   -1.77e+06    1.77e+06
posteam_ATL      0.2000   9.04e+05   2.21e-07      1.000   -1.77e+06    1.77e+06
posteam_BAL      0.3333   9.04e+05   3.69e-07      1.000   -1.77e+06    1.77e+06
posteam_BUF      0.5499   9.04e+05   6.08e-07      1.000   -1.77e+06    1.77e+06
posteam_CAR      0.0177   9.04e+05   1.96e-08      1.000   -1.77e+06    1.77e+06
posteam_CHI      0.2984   9.04e+05    3.3e-07      1.000   -1.77e+06    1.77e+06
posteam_CIN      0.4623   9.04e+05   5.11e-07      1.000   -1.77e+06    1.77e+06
posteam_CLE      0.1849   9.04e+05   2.05e-07      1.000   -1.77e+06    1.77e+06
posteam_DAL      0.4480   9.04e+05   4.95e-07      1.000   -1.77e+06    1.77e+06
posteam_DEN      0.2243   9.04e+05   2.48e-07      1.000   -1.77e+06    1.77e+06
posteam_DET      0.3229   9.04e+05   3.57e-07      1.000   -1.77e+06    1.77e+06
posteam_GB       0.3911   9.04e+05   4.32e-07      1.000   -1.77e+06    1.77e+06
posteam_HOU      0.2439   9.04e+05    2.7e-07      1.000   -1.77e+06    1.77e+06
posteam_IND      0.2025   9.04e+05   2.24e-07      1.000   -1.77e+06    1.77e+06
posteam_JAX      0.2427   9.04e+05   2.68e-07      1.000   -1.77e+06    1.77e+06
posteam_KC       0.5107   9.04e+05   5.65e-07      1.000   -1.77e+06    1.77e+06
posteam_LA       0.2328   9.04e+05   2.57e-07      1.000   -1.77e+06    1.77e+06
posteam_LAC      0.3821   9.04e+05   4.23e-07      1.000   -1.77e+06    1.77e+06
posteam_LV       0.2060   9.04e+05   2.28e-07      1.000   -1.77e+06    1.77e+06
posteam_MIA      0.2439   9.04e+05    2.7e-07      1.000   -1.77e+06    1.77e+06
posteam_MIN      0.2579   9.04e+05   2.85e-07      1.000   -1.77e+06    1.77e+06
posteam_NE       0.1044   9.04e+05   1.15e-07      1.000   -1.77e+06    1.77e+06
posteam_NO       0.2402   9.04e+05   2.66e-07      1.000   -1.77e+06    1.77e+06
posteam_NYG      0.2101   9.04e+05   2.32e-07      1.000   -1.77e+06    1.77e+06
posteam_NYJ      0.1303   9.04e+05   1.44e-07      1.000   -1.77e+06    1.77e+06
posteam_PHI      0.3834   9.04e+05   4.24e-07      1.000   -1.77e+06    1.77e+06
posteam_PIT      0.3427   9.04e+05   3.79e-07      1.000   -1.77e+06    1.77e+06
posteam_SEA      0.1879   9.04e+05   2.08e-07      1.000   -1.77e+06    1.77e+06
posteam_SF       0.5328   9.04e+05   5.89e-07      1.000   -1.77e+06    1.77e+06
posteam_TB       0.3524   9.04e+05    3.9e-07      1.000   -1.77e+06    1.77e+06
posteam_TEN      0.1818   9.04e+05   2.01e-07      1.000   -1.77e+06    1.77e+06
posteam_WAS      0.2847   9.04e+05   3.15e-07      1.000   -1.77e+06    1.77e+06
posteam_type     0.0553      0.024      2.289      0.022       0.008       0.103
defteam_ARI      0.3658   9.04e+05   4.05e-07      1.000   -1.77e+06    1.77e+06
defteam_ATL      0.3688   9.04e+05   4.08e-07      1.000   -1.77e+06    1.77e+06
defteam_BAL      0.2030   9.04e+05   2.25e-07      1.000   -1.77e+06    1.77e+06
d

In [94]:
# print out other classification accuracy metrics
class_report = classification_report(y_test, pred_class)
conf_matrix = confusion_matrix(y_test, pred_class)
roc_auc = roc_auc_score(y_test, pred_prob)
accuracy = accuracy_score(y_test, pred_class)

print(f'Classification Report: \n {class_report}')
print(f'Confusion Matrix: \n {conf_matrix}')
print(f'roc_auc_score: {roc_auc}')
print(f'Accuracy: {accuracy}')

Classification Report: 
               precision    recall  f1-score   support

         0.0       0.69      0.78      0.73      4841
         1.0       0.57      0.45      0.51      3170

    accuracy                           0.65      8011
   macro avg       0.63      0.62      0.62      8011
weighted avg       0.64      0.65      0.64      8011

Confusion Matrix: 
 [[3772 1069]
 [1734 1436]]
roc_auc_score: 0.6853508119721334
Accuracy: 0.6501061041068531


---
## Model with necesary predictors and posteam_type (home field advantage for the offense)

In [95]:
# Get the predictors
X = nfl_data[['ydstogo', 'posteam_ARI', 'posteam_ATL', 'posteam_BAL', 'posteam_BUF',
       'posteam_CAR', 'posteam_CHI', 'posteam_CIN', 'posteam_CLE',
       'posteam_DAL', 'posteam_DEN', 'posteam_DET', 'posteam_GB',
       'posteam_HOU', 'posteam_IND', 'posteam_JAX', 'posteam_KC', 'posteam_LA',
       'posteam_LAC', 'posteam_LV', 'posteam_MIA', 'posteam_MIN', 'posteam_NE',
       'posteam_NO', 'posteam_NYG', 'posteam_NYJ', 'posteam_PHI',
       'posteam_PIT', 'posteam_SEA', 'posteam_SF', 'posteam_TB', 'posteam_TEN',
       'posteam_WAS', 'posteam_type']]

# Get the target predictor
y = nfl_data[['first_down']]

# test train split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Make the model object and train it
model_3 = sm.Logit(y_train, X_train).fit()

# Test the model
pred_prob = model_3.predict(X_test)
pred_class = (pred_prob >= 0.5).astype(int)

# Get the summary
model_3.summary()

Optimization terminated successfully.
         Current function value: 0.615778
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:             first_down   No. Observations:                32040
Model:                          Logit   Df Residuals:                    32006
Method:                           MLE   Df Model:                           33
Date:                Sat, 29 Aug 2026   Pseudo R-squ.:                 0.08343
Time:                        17:13:25   Log-Likelihood:                -19730.
converged:                       True   LL-Null:                       -21525.
Covariance Type:            nonrobust   LLR p-value:                     0.000
================================================================================
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
ydstogo         -0.1596      0.003    -52.358      0.000      -0.166      -0.154
posteam_ARI      0.6663      0.072      9.209      0.000       0.524       0.808
posteam_ATL      0.5079      0.074      6.892      0.000       0.363       0.652
posteam_BAL      0.6159      0.072      8.579      0.000       0.475       0.757
posteam_BUF      0.8328      0.070     11.903      0.000       0.696       0.970
posteam_CAR      0.3204      0.074      4.321      0.000       0.175       0.466
posteam_CHI      0.5814      0.070      8.316      0.000       0.444       0.718
posteam_CIN      0.7306      0.070     10.420      0.000       0.593       0.868
posteam_CLE      0.4682      0.071      6.583      0.000       0.329       0.608
posteam_DAL      0.7530      0.069     10.976      0.000       0.619       0.887
posteam_DEN      0.5226      0.071      7.361      0.000       0.383       0.662
posteam_DET      0.6052      0.073      8.297      0.000       0.462       0.748
posteam_GB       0.6778      0.072      9.415      0.000       0.537       0.819
posteam_HOU      0.5242      0.070      7.466      0.000       0.387       0.662
posteam_IND      0.4810      0.073      6.580      0.000       0.338       0.624
posteam_JAX      0.5238      0.073      7.213      0.000       0.381       0.666
posteam_KC       0.8062      0.069     11.670      0.000       0.671       0.942
posteam_LA       0.5361      0.071      7.580      0.000       0.397       0.675
posteam_LAC      0.6598      0.069      9.505      0.000       0.524       0.796
posteam_LV       0.4753      0.074      6.399      0.000       0.330       0.621
posteam_MIA      0.5485      0.075      7.355      0.000       0.402       0.695
posteam_MIN      0.5536      0.074      7.516      0.000       0.409       0.698
posteam_NE       0.4130      0.073      5.641      0.000       0.269       0.557
posteam_NO       0.5475      0.073      7.503      0.000       0.405       0.691
posteam_NYG      0.5028      0.071      7.059      0.000       0.363       0.642
posteam_NYJ      0.4212      0.075      5.634      0.000       0.275       0.568
posteam_PHI      0.6739      0.068      9.858      0.000       0.540       0.808
posteam_PIT      0.6232      0.069      9.010      0.000       0.488       0.759
posteam_SEA      0.4822      0.075      6.421      0.000       0.335       0.629
posteam_SF       0.8160      0.071     11.429      0.000       0.676       0.956
posteam_TB       0.6619      0.069      9.615      0.000       0.527       0.797
posteam_TEN      0.4780      0.073      6.536      0.000       0.335       0.621
posteam_WAS      0.5672      0.072      7.926      0.000       0.427       0.707
posteam_type     0.0542      0.024      2.246      0.025       0.007       0.102
================================================================================
"""

In [96]:
# Print out other classification accuracy metrics
class_report = classification_report(y_test, pred_class)
conf_matrix = confusion_matrix(y_test, pred_class)
roc_auc = roc_auc_score(y_test, pred_prob)
accuracy = accuracy_score(y_test, pred_class)

print(f'Classification Report: \n {class_report}')
print(f'Confusion Matrix: \n {conf_matrix}')
print(f'roc_auc_score: {roc_auc}')
print(f'Accuracy: {accuracy}')

Classification Report: 
               precision    recall  f1-score   support

         0.0       0.69      0.78      0.73      4841
         1.0       0.57      0.46      0.51      3170

    accuracy                           0.65      8011
   macro avg       0.63      0.62      0.62      8011
weighted avg       0.64      0.65      0.64      8011

Confusion Matrix: 
 [[3763 1078]
 [1718 1452]]
roc_auc_score: 0.6850374397968977
Accuracy: 0.6509799026338784


---

We can see that all have pretty similar accuracy scores, so let's cross validate them to see if there are any differences there.

### Model 1:

In [97]:
# Cross validate model 1

# predictors
X = nfl_data[['ydstogo', 'posteam_ARI', 'posteam_ATL', 'posteam_BAL', 'posteam_BUF',
       'posteam_CAR', 'posteam_CHI', 'posteam_CIN', 'posteam_CLE',
       'posteam_DAL', 'posteam_DEN', 'posteam_DET', 'posteam_GB',
       'posteam_HOU', 'posteam_IND', 'posteam_JAX', 'posteam_KC', 'posteam_LA',
       'posteam_LAC', 'posteam_LV', 'posteam_MIA', 'posteam_MIN', 'posteam_NE',
       'posteam_NO', 'posteam_NYG', 'posteam_NYJ', 'posteam_PHI',
       'posteam_PIT', 'posteam_SEA', 'posteam_SF', 'posteam_TB', 'posteam_TEN',
       'posteam_WAS']]

# Get the target predictor
y = nfl_data[['first_down']]

model = LogisticRegression(max_iter=1000)

# StratifiedKFold keeps class balance consistent across folds — important for classification
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
print(f"Accuracy per fold: {scores}")
print(f"Mean accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

C:\Users\ajhay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\ajhay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\ajhay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the sha

Accuracy per fold: [0.65060542 0.66167291 0.65118602 0.6505618  0.65181024]
Mean accuracy: 0.653 (+/- 0.004)


C:\Users\ajhay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\ajhay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


### Model 2:

In [98]:
# Cross validate model 2

# predictors
X = nfl_data[['ydstogo', 'posteam_ARI', 'posteam_ATL', 'posteam_BAL', 'posteam_BUF',
       'posteam_CAR', 'posteam_CHI', 'posteam_CIN', 'posteam_CLE',
       'posteam_DAL', 'posteam_DEN', 'posteam_DET', 'posteam_GB',
       'posteam_HOU', 'posteam_IND', 'posteam_JAX', 'posteam_KC', 'posteam_LA',
       'posteam_LAC', 'posteam_LV', 'posteam_MIA', 'posteam_MIN', 'posteam_NE',
       'posteam_NO', 'posteam_NYG', 'posteam_NYJ', 'posteam_PHI',
       'posteam_PIT', 'posteam_SEA', 'posteam_SF', 'posteam_TB', 'posteam_TEN',
       'posteam_WAS',
       'posteam_type',
       'defteam_ARI', 'defteam_ATL', 'defteam_BAL',
       'defteam_BUF', 'defteam_CAR', 'defteam_CHI', 'defteam_CIN',
       'defteam_CLE', 'defteam_DAL', 'defteam_DEN', 'defteam_DET',
       'defteam_GB', 'defteam_HOU', 'defteam_IND', 'defteam_JAX', 'defteam_KC',
       'defteam_LA', 'defteam_LAC', 'defteam_LV', 'defteam_MIA', 'defteam_MIN',
       'defteam_NE', 'defteam_NO', 'defteam_NYG', 'defteam_NYJ', 'defteam_PHI',
       'defteam_PIT', 'defteam_SEA', 'defteam_SF', 'defteam_TB', 'defteam_TEN',
       'defteam_WAS']]

# Get the target predictor
y = nfl_data[['first_down']]

model = LogisticRegression(max_iter=1000)

# StratifiedKFold keeps class balance consistent across folds — important for classification
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
print(f"Accuracy per fold: {scores}")
print(f"Mean accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

C:\Users\ajhay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\ajhay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\ajhay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the sha

Accuracy per fold: [0.64860816 0.66117353 0.65181024 0.65006242 0.65131086]
Mean accuracy: 0.653 (+/- 0.004)


### Model 3:

In [99]:
# Cross validate model 3

# predictors
X = nfl_data[['ydstogo', 'posteam_ARI', 'posteam_ATL', 'posteam_BAL', 'posteam_BUF',
       'posteam_CAR', 'posteam_CHI', 'posteam_CIN', 'posteam_CLE',
       'posteam_DAL', 'posteam_DEN', 'posteam_DET', 'posteam_GB',
       'posteam_HOU', 'posteam_IND', 'posteam_JAX', 'posteam_KC', 'posteam_LA',
       'posteam_LAC', 'posteam_LV', 'posteam_MIA', 'posteam_MIN', 'posteam_NE',
       'posteam_NO', 'posteam_NYG', 'posteam_NYJ', 'posteam_PHI',
       'posteam_PIT', 'posteam_SEA', 'posteam_SF', 'posteam_TB', 'posteam_TEN',
       'posteam_WAS',
       'posteam_type']]

# Get the target predictor
y = nfl_data[['first_down']]

model = LogisticRegression(max_iter=1000)

# StratifiedKFold keeps class balance consistent across folds — important for classification
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
print(f"Accuracy per fold: {scores}")
print(f"Mean accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

C:\Users\ajhay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\ajhay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\ajhay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the sha

Accuracy per fold: [0.65035576 0.66067416 0.65043695 0.65143571 0.65218477]
Mean accuracy: 0.653 (+/- 0.004)


Very consistent for all 3 models!

However, due to signifigance of the predictors, 
the chosen final model will be: 

**Model 3** ('ydstogo', 'posteam' dummies, and 'posteam_type').

---

# Conclusion Paragraph:



In order to make a conclusion about the claim: ``all teams in the NFL are equally good at making 3rd and X.'', I should first state the changes I made to the data and the model. Below, I have bulleted the changes I made to the data in terms of column operations and row filtering:

- I imported NFL play by play data for the last 5 seasons (2021 -2025)
- I filtered the data for only 3rd down plays
- I filtered the data so that the following columns did not contain null values: 'ydstogo', 'posteam', 'defteam', 'posteam_type' (this is whether of not the offense is the home team or not), and 'first_down' (binary flag for whether the play resulted in a first down or not).
- After filtering, I mapped the values of 'posteam_type' such that 'home' is now 1 and 'away' is now 0. This will allow us to access whether the offense has home field advantage or not.
- I made dummy variables for offensive ('posteam') and defensive ('defteam') teams on the play.

After applying all of these changes I tested 3 different models using test_train split on a 20% test set as well as additional cross validation where k=5. The first model only included: yards to go until first down ('ydstogo') and offensive/ posession team on the play ('posteam') as discussed in class. Next, I added 2 other potential predictors into the model for model 2: home field advantage for the offense ('posteam_type') and the defensive team on the play ('defteam'), which were also discussed in class. After analyzing the outputs, I made a third model which included: yards to go until first down ('ydstogo'), offensive/ posession team on the play ('posteam') and home field advantage for the offense ('posteam_type'). Although all models produced similar accuacy scores (~65%) on their respectfive tests sets and in their folds during cross validation, model 3 was chosen because all of the predictors were significant and yielded slightly higher accuracy than the other 2 models. Using the output from model 3, we can make a conclusion on the statment: ``all teams in the NFL are equally good at making 3rd and X''.

Using the output from model 3, I would say that the statement ``all teams in the NFL are equally good at making 3rd and X`` is not supported by my results. Every NFL team indicator was a significant predictor in the model, indicating that there is an impact on 3rd down conversion depending on the team. Not only are these indicators significant but their coefficients support this. For example, Kansas City's coefficient (0.8062) is much higher then the Panthers' coefficient (0.3204). In fact, it is over twice as large. This means that when yards to go until first down ('ydstogo') and offensive home field advantage ('posteam_type') are held constant, the log odds of converting on 3rd down are higher for Kansas City than for Carolina. However, it is important to note that the data used in the model only includes the last 5 seasons. The results may look slightly different across a longer range of time due to coaching changes, player changes, etc. 

To conclude, I reject the claim ``that all teams are equally good at making 3rd and X``. 